### INSTALL REQUIRED LIBRARIES

In [0]:
%pip install gspread google-auth pandas

### RETRIEVE DATA THROUGH GOOGLE SHEET API

In [0]:
import gspread
import pandas as pd
import json
from google.oauth2.service_account import Credentials

creds_dict = {
  "type": "---",
  "project_id": "---",
  "private_key_id": "---",
  "private_key": "---",
  "client_id": "---",
  "auth_uri": "---",
  "token_uri": "---",
  "auth_provider_x509_cert_url": "---",
  "client_x509_cert_url": "---",
  "universe_domain": "---"
}


creds = Credentials.from_service_account_info(
    creds_dict,
    scopes=[
        "https://www.googleapis.com/auth/spreadsheets",
        "https://www.googleapis.com/auth/drive"
    ]
)


In [0]:
client = gspread.authorize(creds)
sheet = client.open("STUDENT ATTENDANCE").sheet1

data = sheet.get_all_records()
pdf = pd.DataFrame(data)

display(pdf.head(10))

### TRANSFORM PANDA'S DATAFRAME TO SPARK'S DATAFRAME

In [0]:
from pyspark.sql.functions import current_timestamp

bronze_df = spark.createDataFrame(pdf)
bronze_df = bronze_df.withColumn("ingestion_time", current_timestamp())

display(bronze_df)

### BRONZE TABLE (RAW LOAD)

In [0]:
bronze_df.write.format("delta") \
    .mode("append") \
    .saveAsTable("college_dw.bronze.attendance_raw")

### DISPLAY DATA

### READ DATA FROM BRONZ LAYER

In [0]:
bronze_df = spark.table("college_dw.bronze.attendance_raw")

### CLEAN RAW DATA

In [0]:
from pyspark.sql.functions import col, to_date

silver_df = bronze_df.select(
    col("DATE").alias("attendance_date"),
    col("STUDENT_ID").alias("student_id"),
    col("DEPARTMENT_ID").alias("department_id"),
    col("ingestion_time")
)

silver_df = silver_df.withColumn("attendance_date", to_date("attendance_date"))

### DROP DUPLICATE ATTENDANCE

In [0]:
silver_df = silver_df.dropDuplicates(["student_id", "attendance_date"])

### WRITE SILVER

In [0]:
silver_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("college_dw.silver.attendance_clean")

### GOLD TABLE

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS college_dw.gold.fact_attendance (
    attendance_date DATE,
    student_id STRING,
    department_id STRING,
    ingestion_time TIMESTAMP
)
USING DELTA
PARTITIONED BY (attendance_date)
""")

### INCREMENTAL LOAD

In [0]:
if spark.catalog.tableExists("college_dw.gold.fact_attendance"):

    existing_df = spark.table("college_dw.gold.fact_attendance")

    new_df = silver_df.join(
        existing_df.select("student_id", "attendance_date"),
        on=["student_id", "attendance_date"],
        how="left_anti"
    )

else:
    new_df = silver_df

### WRITE ONLY NEW DATA

In [0]:
new_df.write.format("delta") \
    .mode("append") \
    .saveAsTable("college_dw.gold.fact_attendance")

### VALIDATION

In [0]:
print("New rows inserted:", new_df.count())

display(
    spark.table("college_dw.gold.fact_attendance")
    .orderBy("attendance_date", ascending=False)
    .limit(10)
)

### OPTIMIZE 

In [0]:
spark.sql("""
OPTIMIZE college_dw.gold.fact_attendance
ZORDER BY (student_id)
""")

### VACUUM

In [0]:
spark.sql("""
VACUUM college_dw.gold.fact_attendance RETAIN 168 HOURS
""")